# Exp07 Rough Walk

本实验在粗糙地形上训练带 AMP 风格先验的 G1 速度策略，并比较 height scan 与可选
depth observation。使用 Conda `summer` kernel，并在该环境中用 pip 安装
`mjlab==1.5.0`。提交文件夹只包含
`policy.pt`、`model.py`、`student.py`。

满分 100：提交与有限动作 20；速度跟踪 20；6 m 穿越 20；AMP 10；depth 10；
平滑度 20。`G1_walk_50hz.npz` 源自 humanoid_amp 固定提交并按 BSD-3-Clause
使用，完整哈希和关节顺序见 `assets/motions/manifest.json`，许可文本见
`assets/motions/NOTICE.txt`。

In [1]:
%load_ext autoreload
%autoreload 2

import importlib.util
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Video, display
from tqdm.auto import tqdm


def find_exp_root(name: str) -> Path:
  for candidate in (Path.cwd(), *Path.cwd().parents):
    if candidate.name == name:
      return candidate
    nested = candidate / name
    if nested.is_dir():
      return nested
  raise FileNotFoundError(name)


def load_student(path: Path):
  spec = importlib.util.spec_from_file_location("active_student", path)
  if spec is None or spec.loader is None:
    raise ImportError(path)
  module = importlib.util.module_from_spec(spec)
  spec.loader.exec_module(module)
  return module


#assert Path(sys.prefix).name == "summer", "请切换到 Conda summer kernel"
assert version("mjlab") == "1.5.0"

EXP_ROOT = find_exp_root("exp07_rough_walk")
COURSE_ROOT = EXP_ROOT.parent
STUDENT_FILE = EXP_ROOT / "student.py"
MODE = "height"  # 可改为 "depth"
sys.path.insert(0, str(EXP_ROOT))

from src import workflow  # noqa: E402
from src.mjlab_tasks.env_cfgs import (  # noqa: E402
  course_g1_rough_walk_env_cfg,
)

cfg = course_g1_rough_walk_env_cfg(MODE, student_path=STUDENT_FILE)
assert cfg.auto_reset is False and set(cfg.actions) == {"joint_pos"}
print("环境初始化完成：29 维动作，83 维 AMP state，mode=", MODE)


环境初始化完成：29 维动作，83 维 AMP state，mode= height


## AMP 状态、判别器与风格奖励

单状态拼接 29 维关节位置、29 维关节速度、pelvis 高度、projected gravity、
yaw-local base 线/角速度以及 5 个关键 body 的 pelvis-relative position，共 83 维；
相邻状态组成 166 维判别器输入。

完成 `build_amp_state()`、LSGAN
$L_D=\frac12[(D(s_E)-1)^2+(D(s_\pi)+1)^2]+10L_{gp}$，以及
$r_{style}=\operatorname{clip}(1-0.25(D-1)^2,0,1)$。

In [2]:
s = load_student(STUDENT_FILE)
parts = (
  torch.zeros(2, 29), torch.ones(2, 29), torch.ones(2, 1),
  torch.zeros(2, 3), torch.zeros(2, 3), torch.zeros(2, 3),
  torch.zeros(2, 5, 3),
)
state = s.build_amp_state(*parts)
assert state.shape == (2, 83) and torch.isfinite(state).all()
print("代码检查通过；build_amp_state 对应最终 AMP 10 分的一部分")
loss = s.least_squares_discriminator_loss(
  torch.ones(8), -torch.ones(8), torch.tensor(0.0)
)
torch.testing.assert_close(loss, torch.tensor(0.0))
print("代码检查通过；LSGAN 公式用于训练稳定性")
torch.testing.assert_close(
  s.style_reward(torch.tensor([1.0, 3.0])), torch.tensor([1.0, 0.0])
)
print("代码检查通过；style_reward 对应最终 AMP 10 分的一部分")


代码检查通过；build_amp_state 对应最终 AMP 10 分的一部分
代码检查通过；LSGAN 公式用于训练稳定性
代码检查通过；style_reward 对应最终 AMP 10 分的一部分


## Depth、任务奖励与平滑度

Depth 裁剪到 $[0.1,5.0]$ m 后归一化；任务奖励在零误差时为 1，并随线速度和角速度
误差下降。平滑项使用二阶差分
$p_t=\operatorname{mean}(|a_t-2a_{t-1}+a_{t-2}|)$。完成剩余三个函数。

In [3]:
# s = load_student(STUDENT_FILE)
# depth = s.normalize_depth(torch.tensor([[[[0.0, 0.1, 5.0, 8.0]]]]))
# torch.testing.assert_close(depth, torch.tensor([[[[0.0, 0.0, 1.0, 1.0]]]]))
# print("代码检查通过；normalize_depth 对应可选 depth 10 分")
# torch.testing.assert_close(
#   s.rough_task_reward(torch.zeros(4), torch.zeros(4)), torch.ones(4)
# )
# print("代码检查通过；rough_task_reward 对应任务 40 分")
# actions = torch.ones(3, 29)
# torch.testing.assert_close(
#   s.smoothness_penalty(actions, actions, actions), torch.zeros(3)
# )
# print("代码检查通过；smoothness_penalty 对应平滑度 20 分")


## 环境设计与 smoke

Actor 仅接收 proprioception、command 与 height/depth；critic 保留 privileged
observation。命令课程逐步增加速度与转向范围。下面先显示奖励和课程，再运行
32-env、16-step、强制终止与手动 reset 检查。

In [4]:
# %%time
# display(workflow.plot_training_design(cfg))
# smoke_result = workflow.smoke(
#   MODE, num_envs=32, steps=16, device="cuda:0",
#   student_file=STUDENT_FILE, force_termination=True,
# )
# display(smoke_result)


## AMP-PPO 训练

标准 PPO 每次更新额外进行两次 AMP discriminator 更新。Height 默认 4096 env；
depth 先以 32 env 检查显存。训练结束后 workflow 会从 outputs 中选择最新 checkpoint。

In [ ]:
%%time
TRAIN_ENVS = 64 if MODE == "height" else 32
RUN_DIR = workflow.train(
  MODE, num_envs=TRAIN_ENVS, iterations=600, steps_per_env=24,
  device="cuda:0", student_file=STUDENT_FILE,
)
print(RUN_DIR)


In [6]:
# %%time
# CHECKPOINT = workflow.latest_checkpoint()
# metrics = workflow.evaluate(
#   CHECKPOINT, MODE, num_envs=32, steps=600,
#   device="cuda:0", student_file=STUDENT_FILE,
# )
# display(metrics)
# figure, axis = plt.subplots(figsize=(8, 3.5))
# axis.bar(metrics, metrics.values(), color="#2878b5")
# axis.set_title("Exp07 held-out evaluation")
# axis.grid(axis="y", alpha=0.2)
# display(figure)
# for checkpoint in tqdm([CHECKPOINT], desc="录制 150 帧视频"):
#   video_path = workflow.record_video(
#     checkpoint, MODE, frames=150, device="cuda:0", student_file=STUDENT_FILE
#   )
# display(Video(str(video_path), embed=True))


## 训练结果视频

In [7]:
CHECKPOINT = workflow.latest_checkpoint()
print(f"Checkpoint: {CHECKPOINT}")

video_path = workflow.record_video(
    CHECKPOINT, MODE, frames=150, device="cuda:0", student_file=STUDENT_FILE,
)
print(f"Video saved to: {video_path}")
display(Video(str(video_path), embed=True))

Checkpoint: /home/alan/Desktop/rl/hw/exp07_rough_walk/outputs/rsl_rl/exp07_rough_amp_height/2026-07-21_17-12-35/model_0.pt
Setting seed: 7
Warp 1.15.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 4060 Laptop GPU" (8 GiB, sm_89, mempool enabled)
   Kernel cache:
     /home/alan/Desktop/rl/hw/exp07_rough_walk/outputs/.warp/1.15.0
Module mujoco_warp._src.smooth 91c07ed load on device 'cuda:0' took 2.56 ms  (cached)
Module _nxn_broadphase__locals__kernel_cd21dc9b cd21dc9 load on device 'cuda:0' took 0.38 ms  (cached)
Module ccd_hfield_kernel_builder__locals__ccd_hfield_kernel_7c2ee94b 7c2ee94 load on device 'cuda:0' took 1.53 ms  (cached)
Module ccd_hfield_kernel_builder__locals__ccd_hfield_kernel_67f72f9e 67f72f9 load on device 'cuda:0' took 1.50 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_5f6e1f92 5f6e1f9 load on device 'cuda:0' took 0.95 ms  (cached)
Module mujoco_warp._src.const

In [8]:
# %%time
# submission = workflow.prepare_submission(
#   CHECKPOINT, MODE, device="cuda:0", student_file=STUDENT_FILE
# )
# print("submission:", submission)
# subprocess.run(
#   [sys.executable, str(COURSE_ROOT / "grading_toolkit" / "grade.py"),
#    str(submission), "--task", "exp07", "--device", "cuda:0"],
#   cwd=COURSE_ROOT,
#   check=True,
# )
